# BloodBridge AI — Exploratory Data Analysis

Dataset: Blood Warriors donor/patient dataset (7034 rows, 31 columns)

Explores role distribution, blood group demand, donor reliability metrics, and geographic spread.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.style.use("seaborn-v0_8-whitegrid")

# Load dataset — update path as needed
df = pd.read_csv("/home/dee/Pictures/Dataset.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ── Role Distribution ──────────────────────────────────────────────────────
role_counts = df["role"].value_counts()
print(role_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

role_counts.plot(kind="bar", ax=axes[0], color=["#ef4444","#3b82f6","#8b5cf6","#f59e0b","#10b981"])
axes[0].set_title("Role Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

# Blood group distribution
bg_counts = df["blood_group"].value_counts().head(10)
bg_counts.plot(kind="barh", ax=axes[1], color="#ef4444")
axes[1].set_title("Blood Group Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Patient Analysis ────────────────────────────────────────────────────────
patients = df[df["role"] == "Patient"].copy()
print(f"
Patients: {len(patients)}")
print(f"Bridge blood groups:
{patients["bridge_blood_group"].value_counts()}")

# Transfusion frequency distribution
patients["frequency_in_days"] = pd.to_numeric(patients["frequency_in_days"], errors="coerce")
print(f"
Transfusion frequency (days):
{patients["frequency_in_days"].describe()}")

fig, ax = plt.subplots(figsize=(8, 4))
patients["frequency_in_days"].dropna().hist(bins=20, ax=ax, color="#ef4444", edgecolor="white")
ax.set_title("Patient Transfusion Frequency Distribution", fontsize=13, fontweight="bold")
ax.set_xlabel("Days between transfusions")
plt.tight_layout()
plt.show()

In [ ]:
# ── Donor Reliability Analysis ──────────────────────────────────────────────
donors = df[df["role"].isin(["Bridge Donor", "Emergency Donor"])].copy()
donors["calls_to_donations_ratio"] = pd.to_numeric(donors["calls_to_donations_ratio"], errors="coerce")
donors["donations_till_date"] = pd.to_numeric(donors["donations_till_date"], errors="coerce")

print(f"Donors: {len(donors)}")
print(f"Eligibility status:
{donors["eligibility_status"].value_counts()}")
print(f"
Calls-to-donations ratio:
{donors["calls_to_donations_ratio"].describe()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
donors["calls_to_donations_ratio"].dropna().hist(bins=20, ax=axes[0], color="#3b82f6", edgecolor="white")
axes[0].set_title("Calls-to-Donations Ratio", fontsize=12, fontweight="bold")

donors["donations_till_date"].dropna().hist(bins=20, ax=axes[1], color="#8b5cf6", edgecolor="white")
axes[1].set_title("Total Donations per Donor", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Missing Value Analysis ──────────────────────────────────────────────────
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print(missing_df[missing_df.missing_count > 0].to_string())

# Unknown blood group flag
unknown_bg = df[df["blood_group"] == "Do not Know"]
print(f"
Unknown blood group records: {len(unknown_bg)} ({len(unknown_bg)/len(df)*100:.1f}%)")
print("These will be excluded from matching until blood type is confirmed.")

In [ ]:
# ── Guest Pool Analysis ─────────────────────────────────────────────────────
guests = df[df["role"] == "Guest"].copy()
print(f"Guest accounts: {len(guests)}")
print(f"Guest blood groups:
{guests["blood_group"].value_counts().head(8)}")
print("
Guests are a high-value untapped donor pool for the Guest Activation Campaign.")

## Key EDA Findings

- **7034 total records**: Guests (2420), Emergency Donors (2385), Bridge Donors (2061), Patients (84), Volunteers (83)
- **160 "Do not Know" blood group records** — excluded from matching, flagged for verification
- **O Positive** is the highest demand blood group (1963 donors + highest patient demand)
- **Bombay Blood Group** (2 records) — requires system-wide search and NGO escalation
- **calls_to_donations_ratio** is the strongest predictor of donor reliability
- **2420 Guest accounts** are a major untapped pool for activation campaigns
- **Patients transfuse every 18–26 days** on average — highly predictable demand cycle